# Raport: Analiza uszkodzeń radiacyjnych w białkach i ligandach krystalograficznych

**Autor:** [Twoje Imię i Nazwisko]
**Data:** [Dzisiejsza Data]

## 1. Wstęp teoretyczny
Celem niniejszego ćwiczenia była identyfikacja i analiza uszkodzeń radiacyjnych (ang. *radiation damage*) powstałych w strukturach makromolekularnych pod wpływem promieniowania rentgenowskiego. Analizie poddano dwie struktury krystaliczne:
1.  **3t96**: Receptor AMPA w kompleksie z ligandem Iodo-willardiine.
2.  **4MS4**: Receptor $GABA_B$ w kompleksie z ligandem Baclofen.

Uszkodzenia radiacyjne manifestują się w mapach gęstości elektronowej (szczególnie w mapach różnicowych Fo-Fc) jako obszary ujemnej gęstości (czerwona siatka), wskazujące na utratę atomów, pękanie wiązań lub wzrost nieuporządkowania. Do typowych uszkodzeń należą: dekarboksylacja reszt kwasowych (Asp/Glu), zrywanie mostków dwusiarczkowych oraz odrywanie atomów halogenów w ligandach.

## 2. Metodyka i automatyzacja (Skrypt Python)
Analizę przeprowadzono w środowisku **PyMOL**. W celu usprawnienia procesu wizualizacji i zapewnienia powtarzalności ujęć, przygotowano autorski skrypt w języku Python (damage_scenes.py).

Zgodnie z założeniami, skrypt ten automatycznie generuje sceny dla zdefiniowanej listy uszkodzonych reszt, wykonując następujące operacje:
1.  Pobiera selekcję atomów.
2.  Przybliża widok do konkretnej reszty (zoom).
3.  Generuje mapy 2Fo-Fc (kontur modelu) oraz Fo-Fc (mapa różnicowa uszkodzeń).
4.  Zapisuje widok jako scenę w PyMOL.

Poniżej przedstawiono kod wykorzystanego skryptu:

In [ ]:
from pymol import cmd

def make_damage_scenes(sel_name, obj_name, map_2fofc, map_fofc, around=5.0, level_2fofc=1.0, level_fofc=3.0):
    
    model = cmd.get_model(sel_name)
    seen = set()
    idx = 1
    
    for at in model.atom:
        key = (at.segi, at.chain, at.resi, at.resn)
        if key in seen:
            continue
        seen.add(key)
        sele = f"/{obj_name}//{at.chain}/{at.resi}/"
        
        cmd.hide("everything")
        cmd.show("cartoon", obj_name)
        cmd.show("sticks", sele)
        cmd.util.cnc(sele) 
        cmd.zoom(sele, around)
        
        m2_name = f"m2_{obj_name}_{idx}"
        mfn_name = f"mfn_{obj_name}_{idx}"
        mfp_name = f"mfp_{obj_name}_{idx}"

        cmd.isomesh(m2_name, map_2fofc, level_2fofc, sele, carve=around)
        cmd.color("forest", m2_name)
        
        cmd.isomesh(mfn_name, map_fofc, -level_fofc, sele, carve=around)
        cmd.color("red", mfn_name)
        
        cmd.isomesh(mfp_name, map_fofc, level_fofc, sele, carve=around)
        cmd.color("blue", mfp_name)

        scene_name = f"dmg_{obj_name}_{at.resn}_{at.resi}"
        cmd.scene(scene_name, "store")
        print(f"Zapisano scenę: {scene_name}")
        idx += 1

cmd.extend("make_damage_scenes", make_damage_scenes)

## Wyjaśnienie komend PyMOL

Poniżej przedstawiono szczegółowe wyjaśnienie poleceń wykorzystanych do przygotowania wizualizacji i analizy uszkodzeń radiacyjnych:

* **fetch KOD, type=cif** - Pobiera plik struktury krystalograficznej bezpośrednio z bazy RCSB PDB (np. 3t96). Opcja async=0 jest kluczowa w skryptach, ponieważ wymusza na programie oczekiwanie na pełne pobranie pliku przed wykonaniem kolejnej komendy.
* **load plik.map** - Wczytuje do programu plik znajdujący się na dysku lokalnym. W tym ćwiczeniu używana do załadowania map gęstości elektronowej (formaty .map/.ccp4), których nie można pobrać bezpośrednio przez fetch.
* **isomesh nazwa, mapa, poziom, selekcja** - Generuje trójwymiarową siatkę (izopowierzchnię) reprezentującą gęstość elektronową na zadanym poziomie odcięcia (np. 1.0 $\sigma$ dla mapy 2Fo-Fc lub -3.0 $\sigma$ dla mapy różnicowej). Jest to główne narzędzie do wizualizacji uszkodzeń.
* **carve=2.0** - Parametr komendy isomesh. Ogranicza wyświetlanie siatki mapy tylko do promienia X angstremów (tu: 2.0 Å) wokół wybranych atomów. Pozwala to na "wycięcie" interesującego nas fragmentu mapy i usunięcie szumu z tła, co znacznie poprawia czytelność.
* **select nazwa, kryteria** - Tworzy nazwaną grupę atomów spełniających określone warunki logiczne. W analizie wykorzystywano selektory resn (nazwa reszty, np. IWD) oraz resi (numer reszty), aby wyizolować ligandy i uszkodzone aminokwasy.
* **show reprezentacja, obiekt** - Zmienia sposób wizualizacji struktury. Używano show cartoon dla szkieletu białka oraz show sticks dla ligandów i analizowanych reszt, aby uwidocznić wiązania chemiczne i atomy.
* **hide everything** - Ukrywa wszystkie wyświetlane obiekty (atomy, powierzchnie, wstążki). Używana na początku skryptu, aby "wyczyścić" widok przed nałożeniem nowych reprezentacji.
* **color kolor, obiekt** - Nadaje określony kolor wybranym elementom. Kluczowe dla rozróżnienia map: zielony/niebieski dla gęstości pozytywnej, czerwony dla gęstości negatywnej (uszkodzenia).
* **zoom selekcja** - Centruje kamerę na wybranym obiekcie lub grupie atomów i odpowiednio przybliża widok, co pozwala na szczegółową inspekcję uszkodzenia.
* **bg_color white** - Zmienia kolor tła na biały. Jest to standardowa procedura przy przygotowywaniu publikacji i raportów, zwiększająca kontrast i czytelność zrzutów ekranu.
* **run nazwa_pliku.py** - Wczytuje i wykonuje zewnętrzny skrypt napisany w języku Python. Pozwoliło to na zdefiniowanie własnej funkcji make_damage_scenes i zautomatyzowanie procesu tworzenia scen.

## 3. Analiza uszkodzeń w części białkowej
Wybrano trzy reprezentatywne przykłady uszkodzeń reszt aminokwasowych, zidentyfikowane dzięki silnej ujemnej gęstości na mapach Fo-Fc (poziom -3.0 $\sigma$).4MS4

### Przypadek 1: Dekarboksylacja Glutaminianu (3t96)
**Lokalizacja:** Reszta GLU 125, łańcuch B/D.
![](img/dmg_3t96_2.png)

**Interpretacja:**
Na powyższym obrazie widoczna jest wyraźna ujemna gęstość różnicowa (czerwona siatka) otaczająca grupę karboksylową ($COO^-$) łańcucha bocznego kwasu glutaminowego. Jest to klasyczny marker **dekarboksylacji**. Wysokoenergetyczne fotony powodują zerwanie wiązania węgiel-węgiel i uwolnienie cząsteczki $CO_2$, przez co rzeczywista struktura w krysztale jest krótsza niż zakłada model.

### Przypadek 2: Uszkodzenie Metioniny (3t96)
**Lokalizacja:** Reszta MET 196, łańcuch B/D.
![](img/dmg_3t96_3.png)

**Interpretacja:**
Metionina jest jedną z reszt najbardziej podatnych na uszkodzenia. Widoczna czerwona siatka wokół łańcucha bocznego (szczególnie w okolicach atomu $C_\alpha$ i $C_\beta$) sugeruje, że łańcuch ten uległ znacznemu nieuporządkowaniu lub degradacji. Częstym zjawiskiem jest tu również utlenianie atomu siarki lub zerwanie wiązania $C-S$, co prowadzi do zaniku gęstości w przewidywanym miejscu.

### Przypadek 3: Zerwanie mostka dwusiarczkowego (4MS4)
**Lokalizacja:** Reszta CYS 237, łańcuch B (tworząca mostek z CYS).
![](img/dmg_4ms4_3.png)

**Interpretacja:**
Obraz przedstawia cysteinę z silną ujemną gęstością zlokalizowaną bezpośrednio na atomie siarki ($S_\gamma$). Wskazuje to na radiolityczne **zerwanie wiązania dwusiarczkowego** (Cys-Cys). Elektrony solwatowane i rodniki generowane podczas eksperymentu redukują wiązanie S-S, powodując, że atomy siarki oddalają się od siebie, co mapy różnicowe rejestrują jako "brak atomu" w pierwotnym położeniu.

## 4. Analiza uszkodzeń ligandów
Analizie poddano ligandy w obu strukturach, zwracając szczególną uwagę na atomy chlorowców (Jod, Chlor), które są labilne radiacyjnie.

### Ligand 1: Iodo-willardiine (3t96)
**Lokalizacja:** Reszta IWD 601, atom Iodu (I5).
![](img/dmg_3t96_1.png)

**Interpretacja:**
Ligand Iodo-willardiine zawiera ciężki atom jodu przyłączony do pierścienia uracylu. Na zrzucie ekranu widać masywną czerwoną gęstość (Fo-Fc neg) pokrywającą atom jodu oraz część pierścienia. Świadczy to o **dehalogenacji**, czyli oderwaniu atomu jodu od reszty cząsteczki. Wiązanie C-I jest słabe i łatwo ulega homolizie pod wpływem promieniowania rentgenowskiego, co jest częstym artefaktem w krystalografii.

### Ligand 2: Baclofen (4MS4)
**Lokalizacja:** Reszta 2C0 501 (Baclofen), atom Chloru (CL).
![](img/dmg_4ms4_1.png)

**Interpretacja:**
Podobne zjawisko obserwujemy w przypadku leku Baclofen. Ujemna gęstość zlokalizowana przy atomie chloru (CL) wskazuje na jego częściową utratę lub znaczną ruchliwość/nieuporządkowanie (ang. *disorder*) tego fragmentu cząsteczki. W instrukcji wskazano to jako typowe uszkodzenie ligandów polegające na odrywaniu halogenów lub przesunięciach pierścieni aromatycznych